# Notebook 06: Example 2, many sensors, anomalies in space and time

**Class 03_02.** A small water plant with 8 sensors (flows, level, pump, valve, two temperatures, pressure). They are
driven by two hidden processes, the **demand** (a daily cycle) and the **heat** (a slow wander), so the sensors are
correlated with each other (**space**) and with their own past (**time**). Six days of normal operation train the
detectors; a second plant run of six days carries three events in its second half:

| event | what happens | visible to a per-sensor check? |
|:--|:--|:--|
| **spike** | the level sensor jumps for 30 minutes | yes, out of range |
| **freeze** | the valve sensor repeats a stale value for 6 hours (a stuck or replayed sensor) | no: every value is normal |
| **drift** | the pressure sensor ramps slowly over 10 hours | hardly: small steps |

Detectors: per-sensor z-score; **space only** (one time step, all sensors): PCA reconstruction, Isolation Forest,
dense AE, and *predict each sensor from the others* (the idea of graph-based detectors such as GDN, Deng and Hooi
2021: learn which sensors depend on which, alarm when a sensor departs from what its neighbours predict);
**space and time**: a convolutional AE over 4-hour windows of all 8 sensors.

In [ ]:
import os
import sys

os.environ.setdefault("KERAS_BACKEND", "jax")
import numpy as np
import polars as pl
from matplotlib import pyplot as plt
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import average_precision_score

sys.path.insert(0, ".")
import plots_deep as pdp

import anomlib as an
import deeplib as dl

plt.rcParams["figure.dpi"] = 100
an.table_config()
Xn, _, _ = dl.plant(seed=0)
Xa, ya, events = dl.plant(seed=1, attacks=True)
mu, sd = Xn.mean(0), Xn.std(0)
Xn, Xa = (Xn - mu) / sd, (Xa - mu) / sd
print(pl.DataFrame(events))
print("correlation of the normal sensors (rounded):")
print(np.corrcoef(Xn.T).round(1))

In [ ]:
fig, ax = plt.subplots(4, 1, figsize=(10, 7), sharex=True)
for a, s in zip(ax, [1, 3, 7, 0]):
    a.plot(Xa[:, s], lw=0.7)
    a.set_ylabel(dl.SENSORS[s])
    for e in events:
        a.axvspan(e["start"], e["start"] + e["length"], color="tab:red", alpha=0.15)
ax[-1].set_xlabel("time step (5 min)")
plt.tight_layout()
plt.show()
pdp.plant_events(Xa, events, dl.SENSORS, [1, 3, 7, 0])  # slide plots: assets/figures/plots/03/

## A. Seven detectors, one threshold rule (99.5% quantile of the scores on the normal run)

In [ ]:
def resid_models(X):
    return [LinearRegression().fit(np.delete(X, i, 1), X[:, i]) for i in range(X.shape[1])]


def resid(X, ms):
    return np.max([np.abs(X[:, i] - ms[i].predict(np.delete(X, i, 1))) for i in range(X.shape[1])], 0)


pca = PCA(2).fit(Xn)
ms = resid_models(Xn)
iso = an.IForest().fit(Xn)
ae = dl.AE(latent=2, hidden=16, epochs=30, seed=0).fit(Xn)
W = 48
cae = dl.fit_window_ae(dl.conv_ae(W, 8), dl.windows(Xn, W), epochs=30, seed=0)
DET = {
    "per-sensor max|z|": lambda X: np.abs(X).max(1),
    "PCA recon (space)": lambda X: ((X - pca.inverse_transform(pca.transform(X))) ** 2).sum(1),
    "predict from others (space)": lambda X: resid(X, ms),
    "Isolation Forest (space)": iso.score,
    "dense AE (space)": ae.score,
    "conv AE (space+time)": lambda X: dl.window_to_point(dl.window_error(cae, dl.windows(X, W)), W, len(X)),
}
rng = np.random.default_rng(0)
DET["random score"] = lambda X: rng.random(len(X))
rows, preds = [], {}
for k, f in DET.items():
    s, t = f(Xa), np.quantile(f(Xn), 0.995)
    pred = s > t
    preds[k] = pred
    rows.append({"detector": k, "AUPRC": average_precision_score(ya, s), "F1": dl.f1(pred, ya),
                 "F1 point-adjusted": dl.f1(dl.point_adjust(pred, ya), ya), **dl.events_found(pred, events),
                 "false alarms": int((pred & (ya == 0)).sum())})  # fmt: skip
print(pl.DataFrame(rows))
pdp.space_beats(rows)
pdp.point_adjust(rows)

**Reading.**
* The per-sensor check sees only the spike. The freeze and the drift break the *relation* between sensors, not the
  range of one sensor: they need a model of space (PCA, predict-from-others) or of space and time (conv AE).
* The Isolation Forest and the small dense AE do poorly: with 8 smooth, strongly correlated sensors, the linear
  models (PCA, regression on the other sensors) capture the structure best. **PCA has the best AUPRC and F1** with
  the fewest false alarms.
* The **conv AE over windows** also finds the three events; scored per time step it is behind PCA here. Adding time
  pays off when the anomaly is in the *shape* of the series (notebook 05), less when it is in the relation between
  sensors at each instant.

## B. The point-adjust trap

Many papers on multivariate time series report **point-adjusted** F1: if a single point of an anomalous segment is
flagged, the whole segment counts as detected. Look at the column: a **random score** reaches a point-adjusted F1
of about 0.7, close to the best detectors. Kim et al. (2022) showed this inflation on the standard benchmarks; Wu and
Keogh (2021) showed that several of those benchmarks are solvable by one-liners. Report point-wise metrics, events
found, delay and false alarms, and always include a random and a one-liner baseline.

## C. Which sensor raised the alarm?

The predict-from-others detector names the sensor: the mean residual of each sensor during each event.

In [ ]:
for e in events:
    sl = slice(e["start"], e["start"] + e["length"])
    r = np.array([np.abs(Xa[sl, i] - ms[i].predict(np.delete(Xa[sl], i, 1))).mean() for i in range(8)])
    print(f"{e['kind']:7} true sensor {e['sensor']:9} largest residual: {dl.SENSORS[int(r.argmax())]:9}"
          f" ({r.max():.2f}, next {np.sort(r)[-2]:.2f})")  # fmt: skip

The spike and the freeze are attributed to the right sensor. The drift is not: pressure depends on both hidden
processes, so when it drifts, the prediction of the temperature sensors (which use pressure as an input) also goes
wrong. An attribution is a hint for the analyst, not a proof.